In [1]:
import cv2
import mediapipe as mp
import numpy as np
from ultralytics import YOLO

In [2]:
# Cargar modelo entrenado
model = YOLO("runs/detect/train/weights/best.pt")

# Configurar MediaPipe para detección de manos
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,  # Solo una mano para LSCh
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

# Configurar cámara
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("🚀 Detector LSCh con YOLO + MediaPipe iniciado")
print("Presiona 'q' para salir")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Voltear frame para efecto espejo
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Detectar manos con MediaPipe
    mp_results = hands.process(rgb_frame)
    
    # Variables para tracking
    hand_detected = False
    hand_bbox = None
    
    if mp_results.multi_hand_landmarks:
        for hand_landmarks in mp_results.multi_hand_landmarks:
            # Dibujar landmarks de la mano
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # Obtener bounding box de MediaPipe
            h, w, _ = frame.shape
            x_coords = [lm.x for lm in hand_landmarks.landmark]
            y_coords = [lm.y for lm in hand_landmarks.landmark]
            
            x1, x2 = int(min(x_coords) * w), int(max(x_coords) * w)
            y1, y2 = int(min(y_coords) * h), int(max(y_coords) * h)
            
            # Expandir bbox
            margin = 30
            x1 = max(0, x1 - margin)
            y1 = max(0, y1 - margin)
            x2 = min(w, x2 + margin)
            y2 = min(h, y2 + margin)
            
            hand_bbox = (x1, y1, x2, y2)
            hand_detected = True
            
            # Dibujar bbox de MediaPipe
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)
    
    # Predicción con YOLO
    results = model.predict(
        source=frame,
        conf=0.4,
        verbose=False,
        stream=False
    )
    
    # Procesar resultados
    best_detection = None
    best_score = 0
    
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().numpy()
                cls = int(box.cls[0].cpu().numpy())
                class_name = model.names[cls]
                
                # Verificar overlap con MediaPipe
                overlap_score = 0
                if hand_detected and hand_bbox:
                    mp_x1, mp_y1, mp_x2, mp_y2 = hand_bbox
                    
                    # Calcular intersección
                    inter_x1 = max(x1, mp_x1)
                    inter_y1 = max(y1, mp_y1)
                    inter_x2 = min(x2, mp_x2)
                    inter_y2 = min(y2, mp_y2)
                    
                    if inter_x1 < inter_x2 and inter_y1 < inter_y2:
                        inter_area = (inter_x2 - inter_x1) * (inter_y2 - inter_y1)
                        yolo_area = (x2 - x1) * (y2 - y1)
                        overlap_score = inter_area / yolo_area if yolo_area > 0 else 0
                
                # Calcular score final
                if overlap_score > 0.3:
                    final_score = conf * 1.3  # Boost por MediaPipe
                    color = (0, 255, 255)    # Amarillo
                    verified = True
                else:
                    final_score = conf
                    color = (255, 100, 0)    # Azul
                    verified = False
                
                # Guardar mejor detección
                if final_score > best_score and final_score > 0.3:
                    best_detection = {
                        'bbox': (int(x1), int(y1), int(x2), int(y2)),
                        'class': class_name,
                        'conf': conf,
                        'final_score': final_score,
                        'verified': verified,
                        'color': color
                    }
                    best_score = final_score
    
    # Dibujar detección
    if best_detection:
        x1, y1, x2, y2 = best_detection['bbox']
        
        # Bbox principal
        cv2.rectangle(frame, (x1, y1), (x2, y2), best_detection['color'], 3)
        
        # Letra detectada (grande y centrada)
        letra = best_detection['class']
        precision = best_detection['final_score']
        
        # Texto de la letra (muy grande)
        font_scale = 2.0
        thickness = 3
        (text_width, text_height), _ = cv2.getTextSize(letra, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        
        # Centrar la letra en el bbox
        text_x = x1 + (x2 - x1 - text_width) // 2
        text_y = y1 + (y2 - y1 + text_height) // 2
        
        # Fondo para la letra
        cv2.rectangle(frame, (text_x - 5, text_y - text_height - 5), 
                     (text_x + text_width + 5, text_y + 5), (0, 0, 0), -1)
        
        # Letra en blanco
        cv2.putText(frame, letra, (text_x, text_y), 
                   cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness)
        
        # Precisión arriba del bbox
        precision_text = f"{precision:.1%}"
        cv2.putText(frame, precision_text, (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, best_detection['color'], 2)
        
        # Verificación
        status = "✓ VERIFICADO" if best_detection['verified'] else "SIN VERIFICAR"
        cv2.putText(frame, status, (x1, y2 + 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, best_detection['color'], 2)
    
    # UI en pantalla
    cv2.putText(frame, "Detector LSCh", (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # Status MediaPipe
    mp_status = f"MediaPipe: {'🟢' if hand_detected else '🔴'}"
    cv2.putText(frame, mp_status, (10, 60), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0) if hand_detected else (0, 0, 255), 2)
    
    # Mostrar frame
    cv2.imshow('LSCh - Lenguaje de Señas Chileno', frame)
    
    # Salir con 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Limpiar recursos
cap.release()
cv2.destroyAllWindows()
hands.close()
print("🏁 Detector cerrado")

🚀 Detector LSCh con YOLO + MediaPipe iniciado
Presiona 'q' para salir
🏁 Detector cerrado
🏁 Detector cerrado


In [4]:
# Detector LSCh Estilo YOLO - Cuadro + Letra + Precisión
import cv2
import mediapipe as mp
from ultralytics import YOLO

# Cargar modelo
model = YOLO("runs/detect/train/weights/best.pt")

# MediaPipe para filtrar
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.3
)

# Cámara
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Detector LSCh - Estilo YOLO")
print("'q' para salir")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = cv2.flip(frame, 1)
    
    # Verificar si hay mano
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_results = hands.process(rgb_frame)
    hand_present = mp_results.multi_hand_landmarks is not None
    
    # YOLO solo si hay mano
    if hand_present:
        results = model.predict(
            source=frame,
            conf=0.4,
            verbose=False,
            stream=False
        )
        
        # Procesar detecciones
        for result in results:
            if result.boxes is not None:
                for box in result.boxes:
                    # Obtener datos del bbox
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = float(box.conf[0])
                    cls = int(box.cls[0])
                    letter = model.names[cls]
                    
                    # Solo mostrar si confianza es buena
                    if conf > 0.4:
                        # Convertir a enteros
                        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                        
                        # Color del bbox (verde si alta confianza, amarillo si media)
                        color = (0, 255, 0) if conf > 0.7 else (0, 255, 255)
                        
                        # Dibujar bounding box
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        
                        # Preparar texto
                        label = f"{letter} {conf:.1%}"
                        
                        # Calcular tamaño del texto
                        font = cv2.FONT_HERSHEY_SIMPLEX
                        font_scale = 0.8
                        thickness = 2
                        (text_w, text_h), baseline = cv2.getTextSize(label, font, font_scale, thickness)
                        
                        # Posición del texto (arriba del bbox)
                        text_x = x1
                        text_y = y1 - 10
                        
                        # Si el texto se sale arriba, ponerlo abajo
                        if text_y < text_h:
                            text_y = y2 + text_h + 10
                        
                        # Fondo para el texto
                        cv2.rectangle(frame, 
                                     (text_x, text_y - text_h - 5), 
                                     (text_x + text_w + 5, text_y + 5), 
                                     color, -1)
                        
                        # Texto en negro
                        cv2.putText(frame, label, (text_x + 2, text_y), 
                                   font, font_scale, (0, 0, 0), thickness)
    
    # Mostrar frame
    cv2.imshow('LSCh - YOLO Style', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

Detector LSCh - Estilo YOLO
'q' para salir
